In [1]:
##### Naive Bayes #####
# Why is it called naive? Well, it is naive as it assumes that all of the features are independent, hence the probabilities are independent, i.e. P(X | Y AND Z) = P(X | Y) * P(X | Z)
# How does it differ? It differs by discarding completely the usage of parameters and optimisations. It simply takes a dataset and evaluates probabilites
# How does it improve? It is more resource efficient however it is not as accurate
# It can be Guassian ( assumes the continuous real values are in a normal/Gaussian distrbution )
# or Bernoulli ( we only work with 0/1, F/T, N/Y etc. )
# or it can be Multinomial ( word frequencies for instance )
# it uses at its root the Bayes formula : P(B | A) = P(A | B) * P(B) / P(A)
# with the changed addition : it implements log probs. if we have data with a lot of features, multiplying subunitary values can easily converge towards 0.
# therefore, we must use logs. how does it work? the function used to find probs is already continuous and increasing, therefore applying log keeps this property.
# formula : check https://stefannieuwenhuis.github.io/2025/05/24/math-behind-naive-bayes-part3.html
import numpy as np
import pandas as pd
import kagglehub


dataset, model = None, None
x, y = None, None

def load_data():
    """
    Preprocessing function
    """
    global x, y
    global dataset

    dataset = pd.read_csv("/kaggle/input/datasets/alexteboul/diabetes-health-indicators-dataset/diabetes_012_health_indicators_BRFSS2015.csv")
    dataset = dataset.drop(columns=["Income", "Education", "NoDocbcCost", "AnyHealthcare", "CholCheck"])
    
    y = np.array(dataset["Diabetes_012"])
    x = np.array(dataset.drop(columns="Diabetes_012"))
    

class G_NaiveBayes():
    """
    Gaussian Naive Bayes. Assume all data has independent features and therefore you can compute the probability of each class.
    Finally, when making a prediction, assume each class follows a normal distribution and to predict you take the class
    with most likelihood.
    """
    def __init__(self):
        self.classes, self.priors = [], []
        self.mean, self.var = [], []
        self.eps = 1e-10

    def fit(self, x, y):
        self.classes, counts = np.unique(y, return_counts=True)
        for i, cl in enumerate(self.classes):
            x_class = x[y == cl]
            prob = counts[i]/len(y)
            
            self.priors.append(prob)
            self.mean.append(np.mean(x_class, axis=0))
            self.var.append(np.var(x_class, axis=0))

    
    def _gauss_log_prob(self, sample, classs):
        mean, var = self.mean[classs], self.var[classs] + self.eps
        # we applied log directly to avoid unnecessary computation and underflow
        return np.sum(-0.5 * np.log(2 * np.pi * var) - ((sample - mean)**2) / (2 * var))

    def predict(self, x):
        predictions = []
        for sample in x:
            scores = []
            for clas in range(len(self.classes)):
                score = np.log(self.priors[clas])
                score += self._gauss_log_prob(sample, clas)

                scores.append(score)
            predictions.append(self.classes[np.argmax(scores)])

        return np.array(predictions)

model = G_NaiveBayes()
load_data()
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = G_NaiveBayes()

model.fit(x_train, y_train)

predictions = model.predict(x_test)

print("Accuracy :", accuracy_score(y_test, predictions))
print("Precision:", precision_score(y_test, predictions, average="macro"))
print("Recall   :", recall_score(y_test, predictions, average="macro"))
print("F1 Score :", f1_score(y_test, predictions, average="macro"))

print("\nConfusion Matrix")
print(confusion_matrix(y_test, predictions))

from sklearn.naive_bayes import GaussianNB

clf = GaussianNB()

clf.fit(x_train, y_train)

pred = clf.predict(x_test)

print(confusion_matrix(y_test, pred))
##### Use cases #####
# G NB -> continuous real data ( in R )
# B NB -> binary data
# M NB -> NLP, spam detection etc.

Accuracy : 0.7754257332071902
Precision: 0.41927100781373317
Recall   : 0.44853729023420863
F1 Score : 0.42398010621215326

Confusion Matrix
[[35765    77  6899]
 [  582     3   341]
 [ 3472    23  3574]]
[[35765    77  6899]
 [  582     3   341]
 [ 3472    23  3574]]
